In [ ]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark gdown -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

In [ ]:
import gdown

CATEGORY_NAME = 'Kindle_Store'
FILE_ID = '1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH'
INPUT_PATH = f'/content/{CATEGORY_NAME}.jsonl.gz'

if not os.path.exists(INPUT_PATH):
    gdown.download(id=FILE_ID, output=INPUT_PATH, quiet=False)

OUTPUT_PATH = f'/content/outputs/review_parent_asin_stat_{CATEGORY_NAME}.csv'

Downloading...
From (original): https://drive.google.com/uc?id=1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH
From (redirected): https://drive.google.com/uc?id=1AhlTkSclHtP8IL7LeON0FCcXRln7JvlH&confirm=t&uuid=2d12ac58-313b-40a2-b83c-d95680d2af0c
To: /content/Kindle_Store.jsonl.gz
100%|██████████| 4.57G/4.57G [00:36<00:00, 126MB/s]


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import csv

spark = SparkSession.builder \
    .appName('Asin Counter') \
    .master('local[*]') \
    .config('spark.driver.memory', '40g') \
    .config('spark.python.worker.faulthandler.enabled', 'true') \
    .getOrCreate()

In [ ]:
df = spark.read.json(INPUT_PATH)

counts = df.groupBy('parent_asin').count() \
    .select('parent_asin', col('count')) \
    .coalesce(1) \
    .sort('count', ascending=False)

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
with open(OUTPUT_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['parent_asin', 'count'])
    for row in counts.toLocalIterator():
        writer.writerow([row['parent_asin'], row['count']])

spark.stop()